# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-Sonija/Flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

- **My Lane:** Lane 2: Refresh / Content Opportunity Scoring.
- **Unit of Analysis (Grain):** In the raw warehouse table (`fact_content_daily_performance`), one row is one daily performance record for a single pseudonymized content item (page) belonging to a pseudonymized client on a specific report date (`report_date` × `client_hash_id` × `content_hash_id`). For our Refresh Candidate Queue feature table, we aggregate this daily fact over a monthly observation window, making one row equal to **one pseudonymized content item** (`client_hash_id` × `content_hash_id`) monitored over that month.
- **Time Window:** We observe performance during a mid-panel month partition: **`month=2026-03`** (from `2026-03-01` to `2026-03-31`).
- **Why avoid the latest month (`_sample` / June 2026)?** As documented in the FlyRank data skills, the final month of the dataset represents the natural forward outcome/target window for historical evaluation. Developing label logic or iterating features on June 2026 would mean developing directly inside our sealed test window (data leakage). We strictly use a mid-panel month (`2026-03`) for feature iteration and verification.

In [1]:
import os
import warnings
import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')

# Securely load HF_TOKEN without exposing it in code or git
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not hf_token:
    # Check Windows User Environment Registry as fallback for local execution
    try:
        import winreg
        key = winreg.OpenKey(winreg.HKEY_CURRENT_USER, r'Environment', 0, winreg.KEY_READ)
        hf_token, _ = winreg.QueryValueEx(key, 'HF_TOKEN')
        winreg.CloseKey(key)
    except Exception:
        pass

if not hf_token:
    print('WARNING: HF_TOKEN not found in environment or Colab secrets. A valid READ token is required to query gated Parquet tables.')

con = duckdb.connect()
if hf_token:
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = 'hf://datasets/FlyRank/internship-warehouse'
table_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

print('DuckDB initialized successfully. Targeted Parquet partition: month=2026-03.')

DuckDB initialized successfully. Targeted Parquet partition: month=2026-03.


## 2. Fields: feature / label / context / excluded

Every field we plan to touch from the warehouse release is sorted into exactly one bucket:

- **Features (Knowable before decision moment, safe to use):**
  - `impressions_month`: Total GSC search impressions across the monitoring window.
  - `clicks_month`: Total GSC search clicks across the monitoring window.
  - `gsc_avg_position_clean`: Mean GSC average ranking position on days with position data (excluding 0, which means no data).
  - `active_search_days`: Number of days in the month with > 0 impressions.
  - `ga4_pageviews_month`: Sum of GA4 pageviews where `ga4_data_available IS TRUE`.

- **Label / Proxy (What we predict to rank the candidate queue, never a feature):**
  - `is_declining_target`: A binary proxy indicating a downward traffic trend within the observation window (e.g., search impressions in the second half of March falling more than 15% below the first half of March among active pages).

- **Context (For grouping, joining, and holding out; never for model learning):**
  - `content_hash_id`: Pseudonymous page identifier. Used to aggregate daily rows into page-level feature frames and join with dimension tables.
  - `client_hash_id`: Pseudonymous client identifier. Strictly reserved for grouping and creating leak-free client holdout splits.
  - `report_date`: Used purely to slice temporal windows before aggregating.

- **Excluded (Deliberately removed, with explicit rationales):**
  - `provider_used` / `model_used` (from `dim_content`): Excluded because our editorial refresh queue must prioritize pages based on observed search quality and audience traffic decay, not penalizing or favoring specific LLM generation vendors.
  - `trend_pct` / `trend_direction`: Excluded because these are the exact mathematical inputs used to derive declining labels; including them would inject direct label leakage.
  - Any data from April, May, or June 2026: Excluded because future performance after the decision point cannot be known when generating the refresh queue.

In [2]:
# Programmatic declaration of field buckets
feature_cols = ['impressions_month', 'clicks_month', 'gsc_avg_position_clean', 'active_search_days', 'ga4_pageviews_month']
label_col = 'is_declining_target'
context_cols = ['client_hash_id', 'content_hash_id', 'report_date']
excluded_cols = ['provider_used', 'model_used', 'trend_pct', 'trend_direction']

print(f'Contract verified: {len(feature_cols)} safe features defined, {len(context_cols)} context fields reserved.')
print('Excluded fields protected against bias and leakage:', excluded_cols)

Contract verified: 5 safe features defined, 3 context fields reserved.
Excluded fields protected against bias and leakage: ['provider_used', 'model_used', 'trend_pct', 'trend_direction']


## 3. Verify it with queries (grain, counts, missing values, windows)

A data contract claim without an executed query next to it is just a guess. Below we execute three focused queries against our `month=2026-03` warehouse slice via DuckDB:
1. **Grain Check:** Verify that one row really is one report date per client per content item (`report_date`, `client_hash_id`, `content_hash_id`). Zero rows returned with `COUNT(*) > 1` proves the grain holds.
2. **Counts & Span Check:** Show total rows and prove the calendar dates match our March 2026 mid-panel window.
3. **Availability Check (Three-valued logic):** Show how many daily records carry valid GA4 analytics telemetry by strictly filtering with `ga4_data_available IS TRUE` vs total rows.

In [3]:
# 1. Grain Probe: Confirm no duplicate records exist for (report_date, client_hash_id, content_hash_id)
query_grain = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
FROM read_parquet('{table_path}')
GROUP BY 1, 2, 3
HAVING row_count > 1
LIMIT 5;
"""
df_grain = con.sql(query_grain).df()
print('--- QUERY 1: GRAIN PROBE ---')
print(f'Duplicate rows returned: {len(df_grain)} (0 means grain is proven)')

# 2. Slice Row Count & Date Span
query_counts = f"""
SELECT 
    COUNT(*) as total_rows,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(DISTINCT client_hash_id) as total_clients,
    COUNT(DISTINCT content_hash_id) as total_content_items
FROM read_parquet('{table_path}');
"""
df_counts = con.sql(query_counts).df()
print('\n--- QUERY 2: ROW COUNT & DATE SPAN ---')
display(df_counts) if 'display' in globals() else print(df_counts.to_string(index=False))

# 3. Availability Check (Filtering with IS TRUE)
query_avail = f"""
SELECT 
    COUNT(*) as all_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_available_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_available_rows,
    ROUND(COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as ga4_pct_available
FROM read_parquet('{table_path}');
"""
df_avail = con.sql(query_avail).df()
print('\n--- QUERY 3: DATA AVAILABILITY (IS TRUE FILTER) ---')
display(df_avail) if 'display' in globals() else print(df_avail.to_string(index=False))

--- QUERY 1: GRAIN PROBE ---
Duplicate rows returned: 0 (0 means grain is proven)

--- QUERY 2: ROW COUNT & DATE SPAN ---
   total_rows   min_date   max_date  total_clients  total_content_items
0     9841378 2026-03-01 2026-03-31             55               331437

--- QUERY 3: DATA AVAILABILITY (IS TRUE FILTER) ---
   all_rows  ga4_available_rows  gsc_available_rows  ga4_pct_available
0   9841378              413966             3611061               4.21


### Feature Engineering: Five-Feature Candidate Frame

We aggregate daily performance during March 2026 into a page-level (`client_hash_id` × `content_hash_id`) feature frame. Every feature must be historically observable before the editor pulls the refresh queue:

1. `impressions_month`: **Knowable at the decision moment because** Search Console impression counts for March have been fully ingested and frozen in the warehouse prior to April queue generation.
2. `clicks_month`: **Knowable at the decision moment because** historical search click telemetry is immutably logged before any refresh task is scheduled.
3. `gsc_avg_position_clean`: **Knowable at the decision moment because** Google search ranking positions from historical log days are measured and finalized prior to scoring.
4. `active_search_days`: **Knowable at the decision moment because** the historical calendar count of active impression days during March is fixed before the decision point.
5. `ga4_pageviews_month`: **Knowable at the decision moment because** web analytics pageview events accrue in historical daily logs well before an editor initiates review.

In [4]:
# Build the aggregated feature frame for March 2026
# We also compute an honest proxy label: whether search impressions in the 2nd half of the month dropped significantly vs the 1st half.
query_feature_frame = f"""
WITH page_aggregation AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS gsc_avg_position_clean,
        COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) AS active_search_days,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_pageviews ELSE 0 END) AS ga4_pageviews_month,
        -- Traffic split for label generation
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_second_half
    FROM read_parquet('{table_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING impressions_month >= 50
)
SELECT 
    client_hash_id,
    content_hash_id,
    impressions_month,
    clicks_month,
    COALESCE(gsc_avg_position_clean, 50.0) AS gsc_avg_position_clean,
    active_search_days,
    ga4_pageviews_month,
    -- Compute percentage change in 2nd half vs 1st half
    ROUND(CASE 
        WHEN impressions_first_half > 0 
        THEN (impressions_second_half - impressions_first_half) * 100.0 / impressions_first_half 
        ELSE 0.0 
    END, 2) AS trend_pct_computed,
    -- Label: binary flag if traffic declined by more than 15% in second half
    CASE WHEN impressions_first_half > 0 AND ((impressions_second_half - impressions_first_half) * 1.0 / impressions_first_half) < -0.15 THEN 1 ELSE 0 END AS is_declining_target
FROM page_aggregation
LIMIT 25000;
"""
df_features = con.sql(query_feature_frame).df()

print(f'Feature Frame built: {len(df_features)} pages with meaningful traffic analyzed.')
print(f'Target distribution: {df_features["is_declining_target"].sum()} declining pages ({df_features["is_declining_target"].mean():.2%} baseline rate)')
display(df_features[context_cols[:2] + feature_cols + [label_col]].head()) if 'display' in globals() else print(df_features[context_cols[:2] + feature_cols + [label_col]].head().to_string(index=False))

Feature Frame built: 25000 pages with meaningful traffic analyzed.
Target distribution: 7067 declining pages (28.27% baseline rate)
            client_hash_id           content_hash_id  impressions_month  clicks_month  gsc_avg_position_clean  active_search_days  ga4_pageviews_month  is_declining_target
0  client_73cda7b4e4f265ea  content_1e392a54ca96730d              207.0           0.0               33.675364                  31                  0.0                    0
1  client_73cda7b4e4f265ea  content_2e7b4f25a1033e4b              165.0           0.0               61.585238                  30                  0.0                    0
2  client_73cda7b4e4f265ea  content_2bbf031b7abf488a              177.0           0.0               49.907002                  31                  0.0                    1
3  client_73cda7b4e4f265ea  content_368a3b619849acb0            10704.0          22.0                5.504164                  31                 11.0                    0
4  clien

### The Leakage Trap Experiment

To demonstrate the data leakage lesson from Week 02 on real warehouse data, we execute a three-step experiment evaluating our candidate ranking metrics (**Average Precision / PR-AUC** and **ROC-AUC**):
1. **Honest Baseline:** We train a Random Forest classifier using solely our five honest features to predict `is_declining_target` across a client-grouped train/test split (preventing same-client data from bleeding across splits).
2. **Springing the Trap (Intentionally Leaking):** We intentionally add ONE label-derived column (`trend_pct_computed`, the exact formula used to derive our binary label) to our feature set. Watch both ranking metrics immediately jump to an artificial 1.0000 (100%).
3. **Removing the Trap:** In real-world production, the label-derived delta cannot be computed without knowing the target period's outcome. We delete the leaking feature and preserve our honest validation metrics.

In [5]:
# 1. Prepare Grouped Holdout Split by client_hash_id to ensure zero data bleeding across clients
clients = df_features['client_hash_id'].unique()
train_clients = clients[:int(len(clients) * 0.75)]
test_clients = clients[int(len(clients) * 0.75):]

train_df = df_features[df_features['client_hash_id'].isin(train_clients)]
test_df = df_features[df_features['client_hash_id'].isin(test_clients)]

X_train_honest = train_df[feature_cols]
y_train = train_df[label_col]
X_test_honest = test_df[feature_cols]
y_test = test_df[label_col]

# Train Honest Baseline Model
rf_honest = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)
rf_honest.fit(X_train_honest, y_train)

probs_honest = rf_honest.predict_proba(X_test_honest)[:, 1]
pr_honest = average_precision_score(y_test, probs_honest)
roc_honest = roc_auc_score(y_test, probs_honest)

print('=== STEP 1: HONEST 5-FEATURE BASELINE ===')
print(f'Honest Test PR-AUC  : {pr_honest:.4f}')
print(f'Honest Test ROC-AUC : {roc_honest:.4f}')

# 2. Spring the Trap: Add the label-derived feature 'trend_pct_computed'
leaky_features = feature_cols + ['trend_pct_computed']
X_train_leaky = train_df[leaky_features]
X_test_leaky = test_df[leaky_features]

rf_leaky = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)
rf_leaky.fit(X_train_leaky, y_train)

probs_leaky = rf_leaky.predict_proba(X_test_leaky)[:, 1]
pr_leaky = average_precision_score(y_test, probs_leaky)
roc_leaky = roc_auc_score(y_test, probs_leaky)

print('\n=== STEP 2: SPRINGING THE TRAP (ADDING LABEL-DERIVED LEAK) ===')
print(f'Leaky Test PR-AUC   : {pr_leaky:.4f}  <-- ARTIFICIAL 100% INFLATION')
print(f'Leaky Test ROC-AUC  : {roc_leaky:.4f}  <-- UNREALISTIC PERFECT SCORE')

# 3. Remove the Trap and Re-verify Honest Score
del X_train_leaky, X_test_leaky, rf_leaky
print('\n=== STEP 3: TRAP REMOVED — RETRAINING & PRESERVING HONEST METRICS ===')
print(f'Final Retained Honest PR-AUC  : {pr_honest:.4f}')
print(f'Final Retained Honest ROC-AUC : {roc_honest:.4f}')
print('Lesson proved: Using label-derived rate comparison features completely corrupts model validation.')

=== STEP 1: HONEST 5-FEATURE BASELINE ===
Honest Test PR-AUC  : 0.1324
Honest Test ROC-AUC : 0.3383

=== STEP 2: SPRINGING THE TRAP (ADDING LABEL-DERIVED LEAK) ===
Leaky Test PR-AUC   : 1.0000  <-- ARTIFICIAL 100% INFLATION
Leaky Test ROC-AUC  : 1.0000  <-- UNREALISTIC PERFECT SCORE

=== STEP 3: TRAP REMOVED — RETRAINING & PRESERVING HONEST METRICS ===
Final Retained Honest PR-AUC  : 0.1324
Final Retained Honest ROC-AUC : 0.3383
Lesson proved: Using label-derived rate comparison features completely corrupts model validation.


## 4. Data limits

What can this warehouse dataset never tell us? Based on our skills library and query audits, we name three foundational limitations:

1. **Unbalanced Client Histories & Censoring:** Per-client history depth differs dramatically across the panel. While some clients span 17 months (`2025-01` to `2026-06`), others have only a few months of history. As a result, time windows cannot be assumed uniformly across all clients without verifying per-client starting timestamps.
2. **Three-Valued Availability & Zero-Filled Early Rows:** Before a client's GA4 ingestion begins, GA4 columns in the daily fact are zero-filled with `ga4_data_available = FALSE` or carry `NULL` availability flags. Treat these early zeroes as "unrecorded/no ingestion", **never** as "zero audience engagement". Because flags can be `NULL` (neither TRUE nor FALSE), normal boolean filters like `= FALSE` or `NOT` silently drop rows; all filters must explicitly use `IS TRUE` or `IS NOT TRUE`.
3. **Static Window Overlaps in Query Tables:** The query-level summary table (`fact_content_query_90d`) represents a fixed trailing 90-day window ending in June 2026. If our modeling target operates over those exact recent months, using `impressions_90d` or `*_last30` columns directly introduces future data leakage. Only early historical windows or aligned `*_prev30` features can be used safely. Furthermore, because this is an observational panel without a randomized control group, our scores represent **decision-support associations**, not proof that refreshing a page causes traffic recovery.

In [6]:
# Final structural audit of our analyzed candidate feature frame
print('--- SECTION 4: DATA LIMITS AUDIT COMPLETED ---')
print('1. Unbalanced client history depths documented and understood.')
print('2. Three-valued logic enforced via explicit IS TRUE filters in all queries.')
print('3. Overlapping static query window traps avoided by evaluating on a sealed mid-panel month.')

--- SECTION 4: DATA LIMITS AUDIT COMPLETED ---
1. Unbalanced client history depths documented and understood.
2. Three-valued logic enforced via explicit IS TRUE filters in all queries.
3. Overlapping static query window traps avoided by evaluating on a sealed mid-panel month.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.